# ⚠️ NOTEBOOK ARQUIVADO

**Este notebook foi incorporado ao pipeline principal:**

- **Análise espectral (FFT, assinaturas por classe)** → `01_EDA.ipynb` — Seção B
- **Features espectrais e resistentes a deriva** → `02_Feature_Engineering.ipynb` — Seção B

Manter como referência histórica. **Não executar em produção.**

---


# 04 — Análise Espectral 100 Hz | 7 Classes

Pipeline completo de análise espectral para calibração de bandas e treino do modelo GNB de 7 classes.

**Coleta**: 100 Hz, janela 1000 amostras (10 s) → resolução Δf = 0.1 Hz  
**Pipeline FFT**: detrend linear → janela Hann → zero-padding n_fft=4096 → bins a 0.024 Hz  
**Classes**: FAN_OFF · LOW_ROT_OFF · LOW_ROT_ON · MEDIUM_ROT_OFF · MEDIUM_ROT_ON · HIGH_ROT_OFF · HIGH_ROT_ON

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'shared'))

import numpy as np
import pandas as pd
import json
import time
import oracledb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.stats import f_oneway

from feature_engineering import (
    compute_spectral_signature,
    compute_spectral_features,
    compute_time_features,
    compute_drift_resistant_features,
    spectral_moments_p1_p14,
    compute_spectral_moments_features,
    extract_features_windowed_spectral,
    extract_features_windowed,
    extract_features_windowed_drift_resistant,
    extract_features_windowed_spectral_moments,
    SPECTRAL_BANDS,
)

# ── Parâmetros da coleta ──────────────────────────────────────────────────────
COLLECTION_ID = "col_20260223_115149_100hz"
SAMPLE_HZ     = 100
WINDOW_SIZE   = 1000    # 10 s → Δf = 0.1 Hz
STEP_SIZE     = 500     # 50 % overlap → nova análise a cada 5 s
N_FFT         = 4096    # zero-padding → bins a 0.024 Hz

AXES = ['accel_x_g', 'accel_y_g', 'accel_z_g',
        'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps']

CLASSES_ORDER = [
    'FAN_OFF',
    'LOW_ROT_OFF',  'LOW_ROT_ON',
    'MEDIUM_ROT_OFF','MEDIUM_ROT_ON',
    'HIGH_ROT_OFF', 'HIGH_ROT_ON',
]

COMPOSITE_MAP = {
    ('OFF',    'STOPPED'):  'FAN_OFF',
    ('LOW',    'STOPPED'):  'LOW_ROT_OFF',
    ('LOW',    'ROTATING'): 'LOW_ROT_ON',
    ('MEDIUM', 'STOPPED'):  'MEDIUM_ROT_OFF',
    ('MEDIUM', 'ROTATING'): 'MEDIUM_ROT_ON',
    ('HIGH',   'STOPPED'):  'HIGH_ROT_OFF',
    ('HIGH',   'ROTATING'): 'HIGH_ROT_ON',
}

# ROT_ON  → linha sólida  (cor forte, primária)
# ROT_OFF → linha tracejada (cor contrastante, sem transparência)
COLORS = {
    'FAN_OFF':        '#7f7f7f',  # cinza    — sólida
    'LOW_ROT_ON':     '#1f77b4',  # azul     — sólida
    'LOW_ROT_OFF':    '#ff7f0e',  # laranja  — tracejada
    'MEDIUM_ROT_ON':  '#2ca02c',  # verde    — sólida
    'MEDIUM_ROT_OFF': '#9467bd',  # roxo     — tracejada
    'HIGH_ROT_ON':    '#d62728',  # vermelho — sólida
    'HIGH_ROT_OFF':   '#17becf',  # ciano    — tracejada
}

print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"SAMPLE_HZ     : {SAMPLE_HZ} Hz  |  Nyquist: {SAMPLE_HZ/2} Hz")
print(f"WINDOW_SIZE   : {WINDOW_SIZE} pts = {WINDOW_SIZE/SAMPLE_HZ:.0f} s")
print(f"N_FFT         : {N_FFT}  →  bin spacing: {SAMPLE_HZ/N_FFT:.4f} Hz")

COLLECTION_ID : col_20260223_115149_100hz
SAMPLE_HZ     : 100 Hz  |  Nyquist: 50.0 Hz
WINDOW_SIZE   : 1000 pts = 10 s
N_FFT         : 4096  →  bin spacing: 0.0244 Hz


## 1. Carga de Dados — Oracle (transition_marker = 0)

In [2]:
conn = oracledb.connect(user="dersao", password="986960440", dsn="localhost:1521/xepdb1")

sql = """
    SELECT ts_epoch, accel_x_g, accel_y_g, accel_z_g,
           gyro_x_dps, gyro_y_dps, gyro_z_dps,
           cmd_speed_label, rot_state_label, transition_marker, sample_rate
    FROM sensor_data
    WHERE collection_id    = :cid
      AND transition_marker = 0
    ORDER BY ts_epoch ASC
"""
df_raw = pd.read_sql(sql, conn, params={"cid": COLLECTION_ID})
conn.close()

# Normalizar nomes de colunas (Oracle retorna em maiúsculas)
df_raw.columns = [c.lower() for c in df_raw.columns]

# Reconstruir label composto
df_raw['fan_state'] = df_raw.apply(
    lambda r: COMPOSITE_MAP.get(
        (str(r['cmd_speed_label']).upper(), str(r['rot_state_label']).upper()),
        'UNKNOWN'
    ), axis=1
)

# ts_epoch é epoch em segundos (NUMBER 18,6)
df_raw['timestamp_s'] = df_raw['ts_epoch'].astype(float)

print(f"Total amostras (transition=0): {len(df_raw):,}")
print(f"\nAmostras por classe:")
vc = df_raw['fan_state'].value_counts()
for cls in CLASSES_ORDER:
    n = vc.get(cls, 0)
    dur = n / SAMPLE_HZ
    print(f"  {cls:<20} {n:>7,} amostras  ({dur:.0f} s)")

Total amostras (transition=0): 0

Amostras por classe:
  FAN_OFF                    0 amostras  (0 s)
  LOW_ROT_OFF                0 amostras  (0 s)
  LOW_ROT_ON                 0 amostras  (0 s)
  MEDIUM_ROT_OFF             0 amostras  (0 s)
  MEDIUM_ROT_ON              0 amostras  (0 s)
  HIGH_ROT_OFF               0 amostras  (0 s)
  HIGH_ROT_ON                0 amostras  (0 s)


C:\Users\Anderson\AppData\Local\Temp\ipykernel_14260\391712853.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(sql, conn, params={"cid": COLLECTION_ID})


## 2. Assinaturas Espectrais (FFT médio por classe)

Cada curva = média de N janelas de 10 s.  
Picos dominantes identificam a frequência característica de cada classe.  
**Use este gráfico para calibrar os SPECTRAL_BANDS.**

In [3]:
def mean_spectrum(df_class, axis, n_windows=20):
    """Média das assinaturas espectrais de n_windows janelas."""
    n = len(df_class)
    if n < WINDOW_SIZE:
        return None, None
    starts = np.linspace(0, n - WINDOW_SIZE,
                         min(n_windows, (n - WINDOW_SIZE) // STEP_SIZE + 1),
                         dtype=int)
    all_mags = []
    for s in starts:
        vals = df_class[axis].iloc[s:s + WINDOW_SIZE].values
        freqs, mags, _, _, _ = compute_spectral_signature(
            vals, sampling_hz=SAMPLE_HZ, n_fft=N_FFT)
        all_mags.append(mags)
    return freqs, np.mean(all_mags, axis=0)

os.makedirs('output/figures', exist_ok=True)

# ── Estilo por estado de rotação ──────────────────────────────────────────────
#   ROT_ON  → linha sólida  (2.0 px)
#   ROT_OFF → linha tracejada (2.2 px, mesma legibilidade)
def line_style(cls):
    is_off = cls.endswith('ROT_OFF')
    return dict(color=COLORS[cls],
                width=2.2 if is_off else 2.0,
                dash='dash' if is_off else 'solid')

# ── Rótulos curtos para botões individuais ────────────────────────────────────
SHORT_LABELS = {
    'FAN_OFF':       'FAN OFF',
    'LOW_ROT_ON':    'LO ↑',
    'LOW_ROT_OFF':   'LO ↓',
    'MEDIUM_ROT_ON': 'MD ↑',
    'MEDIUM_ROT_OFF':'MD ↓',
    'HIGH_ROT_ON':   'HI ↑',
    'HIGH_ROT_OFF':  'HI ↓',
}

# ── Passos dos sliders ────────────────────────────────────────────────────────
X_MIN_STEPS = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
Y_MIN_STEPS = [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]

for axis in AXES:
    fig = go.Figure()
    peak_table   = []
    added_classes = []

    n_loaded = {cls: (df_raw['fan_state'] == cls).sum() for cls in CLASSES_ORDER}
    if all(n == 0 for n in n_loaded.values()):
        print(f"  AVISO [{axis}]: df_raw sem dados — execute a célula de carga primeiro")
        continue

    for cls in CLASSES_ORDER:
        df_cls = df_raw[df_raw['fan_state'] == cls].reset_index(drop=True)
        freqs, avg_mags = mean_spectrum(df_cls, axis)
        if freqs is None:
            continue

        added_classes.append(cls)

        mask   = (freqs >= 0.01) & (freqs <= 50.0)
        f_show = freqs[mask]
        m_show = avg_mags[mask]

        mask_peak = f_show >= 0.5
        if mask_peak.any():
            idx_pk = int(np.argmax(m_show[mask_peak]))
            peak_f = float(f_show[mask_peak][idx_pk])
            peak_m = float(m_show[mask_peak][idx_pk])
        else:
            peak_f, peak_m = 0.0, 0.0

        peak_table.append({'Classe': cls,
                           'Pico Hz': round(peak_f, 3),
                           'Magnitude': round(peak_m, 6)})

        fig.add_trace(go.Scatter(
            x=f_show, y=m_show,
            name=cls,
            line=line_style(cls),
            hovertemplate=(f'<b>{cls}</b><br>'
                           'f=%{x:.3f} Hz<br>mag=%{y:.6f}<extra></extra>'),
        ))

    # ── Sombreamento das bandas ────────────────────────────────────────────────
    for band, (flo, fhi) in SPECTRAL_BANDS.items():
        if fhi <= 50.0:
            fig.add_vrect(x0=flo, x1=fhi,
                          fillcolor='rgba(180,180,180,0.10)', line_width=0,
                          annotation_text=band, annotation_position='top left',
                          annotation_font_size=10)

    # ── Helpers de visibilidade ───────────────────────────────────────────────
    def vis(subset):
        return [c in subset for c in added_classes]

    all_set = set(added_classes)
    rot_on  = {'LOW_ROT_ON',  'MEDIUM_ROT_ON',  'HIGH_ROT_ON'}
    rot_off = {'LOW_ROT_OFF', 'MEDIUM_ROT_OFF', 'HIGH_ROT_OFF'}
    baixa   = {'LOW_ROT_ON',    'LOW_ROT_OFF'}
    media   = {'MEDIUM_ROT_ON', 'MEDIUM_ROT_OFF'}
    alta    = {'HIGH_ROT_ON',   'HIGH_ROT_OFF'}

    # ── Botões de grupo ───────────────────────────────────────────────────────
    group_buttons = [
        dict(label='Todas',   method='restyle', args=[{'visible': vis(all_set)}]),
        dict(label='ROT ON',  method='restyle', args=[{'visible': vis(rot_on)}]),
        dict(label='ROT OFF', method='restyle', args=[{'visible': vis(rot_off)}]),
        dict(label='Baixa',   method='restyle', args=[{'visible': vis(baixa)}]),
        dict(label='Média',   method='restyle', args=[{'visible': vis(media)}]),
        dict(label='Alta',    method='restyle', args=[{'visible': vis(alta)}]),
    ]

    # ── Botões individuais ────────────────────────────────────────────────────
    individual_buttons = [
        dict(label=SHORT_LABELS.get(cls, cls),
             method='restyle',
             args=[{'visible': vis({cls})}])
        for cls in added_classes
    ]

    # ── Sliders (abaixo do gráfico) ───────────────────────────────────────────
    x_min_slider = dict(
        active=0,
        currentvalue=dict(prefix='X min: ', suffix=' Hz', visible=True, xanchor='left'),
        pad=dict(t=35, b=5, l=10, r=10),
        len=0.47, x=0.0, y=-0.26,
        steps=[
            dict(label=f'{v:.2g} Hz', method='relayout',
                 args=[{'xaxis.type': 'log',
                        'xaxis.range': [np.log10(v), np.log10(50.0)]}])
            for v in X_MIN_STEPS
        ],
    )
    y_min_slider = dict(
        active=1,
        currentvalue=dict(prefix='Y min: ', visible=True, xanchor='left'),
        pad=dict(t=35, b=5, l=10, r=10),
        len=0.47, x=0.53, y=-0.26,
        steps=[
            dict(label=f'{v:.0e}', method='relayout',
                 args=[{'yaxis.type': 'log',
                        'yaxis.range[0]': np.log10(v)}])
            for v in Y_MIN_STEPS
        ],
    )

    # ── Layout: botões acima do título → título → legenda → gráfico → sliders ─
    #
    #  [Individuais  y=1.82]   FAN OFF | LO ↑ | LO ↓ | MD ↑ | MD ↓ | HI ↑ | HI ↓
    #  [Grupos       y=1.56]   Todas | ROT ON | ROT OFF | Baixa | Média | Alta
    #  [Escala  Y    y=1.30]   Y linear | Y log      [Escala X]  X linear | X log
    #  TÍTULO  (container y≈0.59 = justo acima da área de plot)
    #  ─────── PLOT AREA ─────────────────────────────────────  LEGENDA →
    #  [Slider X min]   [Slider Y min]
    fig.update_layout(
        # ── Título abaixo dos botões ──────────────────────────────────────────
        title=dict(
            text=f'Assinaturas Espectrais — {axis}  '
                 f'(100 Hz · Hann · zero-pad | ─ ROT ON  ╌ ROT OFF)',
            x=0.0, xanchor='left',
            y=0.59, yanchor='bottom',  # container coords ≈ paper y=1.08
            font=dict(size=12),
        ),
        xaxis_title='Frequência (Hz)',
        yaxis_title='Magnitude',
        hovermode='x unified',
        height=720,
        margin=dict(t=310, b=185, r=155, l=60),
        # ── Legenda vertical à direita ────────────────────────────────────────
        legend=dict(
            x=1.01, y=1.0,
            xanchor='left', yanchor='top',
            orientation='v',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='rgba(0,0,0,0.15)',
            borderwidth=1,
        ),
        sliders=[x_min_slider, y_min_slider],
        updatemenus=[
            # ── Linha 1: escala Y ──────────────────────────────────────────
            dict(
                type='buttons', direction='left',
                x=0.0, y=1.30, xanchor='left', yanchor='top',
                showactive=True, pad={'r': 6, 't': 4},
                buttons=[
                    dict(label='Y linear', method='relayout',
                         args=[{'yaxis.type': 'linear'}]),
                    dict(label='Y log',    method='relayout',
                         args=[{'yaxis.type': 'log'}]),
                ],
            ),
            # ── Linha 1: escala X ──────────────────────────────────────────
            dict(
                type='buttons', direction='left',
                x=0.28, y=1.30, xanchor='left', yanchor='top',
                showactive=True, pad={'r': 6, 't': 4},
                buttons=[
                    dict(label='X linear', method='relayout',
                         args=[{'xaxis.type': 'linear',
                                'xaxis.range': [0.01, 50]}]),
                    dict(label='X log',    method='relayout',
                         args=[{'xaxis.type': 'log',
                                'xaxis.range': [np.log10(0.01), np.log10(50)]}]),
                ],
            ),
            # ── Linha 2: grupos de classes ─────────────────────────────────
            dict(
                type='buttons', direction='left',
                x=0.0, y=1.56, xanchor='left', yanchor='top',
                showactive=True, pad={'r': 4, 't': 4},
                buttons=group_buttons,
            ),
            # ── Linha 3: classes individuais ───────────────────────────────
            dict(
                type='buttons', direction='left',
                x=0.0, y=1.82, xanchor='left', yanchor='top',
                showactive=True, pad={'r': 4, 't': 4},
                buttons=individual_buttons,
            ),
        ],
    )

    fig.write_html(f'output/figures/04_spectrum_{axis}.html')
    fig.show()

    if peak_table:
        df_peaks = pd.DataFrame(peak_table)
        print(f"\n{'─'*55}")
        print(f"  {axis} — picos dominantes:")
        print(df_peaks[['Classe', 'Pico Hz', 'Magnitude']].to_string(index=False))


  AVISO [accel_x_g]: df_raw sem dados — execute a célula de carga primeiro
  AVISO [accel_y_g]: df_raw sem dados — execute a célula de carga primeiro
  AVISO [accel_z_g]: df_raw sem dados — execute a célula de carga primeiro
  AVISO [gyro_x_dps]: df_raw sem dados — execute a célula de carga primeiro
  AVISO [gyro_y_dps]: df_raw sem dados — execute a célula de carga primeiro
  AVISO [gyro_z_dps]: df_raw sem dados — execute a célula de carga primeiro


## 3. Calibração dos SPECTRAL_BANDS

Ajuste os limites abaixo com base nos picos observados no gráfico acima.  
Execute a célula para atualizar `feature_engineering.py` automaticamente.

In [4]:
# ── EDITE AQUI após olhar os gráficos ────────────────────────────────────────
# Substitua pelos picos reais observados (±50 % de margem ao redor do pico)
BANDS_CALIBRATED = {
    'rot':   (0.10,  1.50),   # rotação do suporte (~0.25 Hz = 1 rot/4s) — AJUSTAR
    'vel1':  (3.00,  6.50),   # LOW  speed (~5 Hz) — AJUSTAR
    'vel2':  (6.50,  9.50),   # MEDIUM speed (~7.5 Hz) — AJUSTAR
    'vel3':  (9.50, 14.00),   # HIGH speed (~10–12 Hz) — AJUSTAR
    'noise': (25.0,  50.0),   # noise floor (referência)
}

print("Bandas atuais vs calibradas:")
for name in BANDS_CALIBRATED:
    old = SPECTRAL_BANDS.get(name, ('?','?'))
    new = BANDS_CALIBRATED[name]
    changed = "← ALTERADO" if old != new else ""
    print(f"  {name:<8}  atual={old}  novo={new}  {changed}")

# ── Confirmar antes de salvar ─────────────────────────────────────────────────
# Descomente a linha abaixo APÓS confirmar os valores com os gráficos:
# _SAVE_BANDS = True
_SAVE_BANDS = False  # mude para True quando os valores estiverem corretos

if _SAVE_BANDS:
    fe_path = os.path.join('shared', 'feature_engineering.py')
    with open(fe_path, 'r', encoding='utf-8') as f:
        src = f.read()

    import re
    new_bands_str = "SPECTRAL_BANDS = {\n"
    for name, (flo, fhi) in BANDS_CALIBRATED.items():
        new_bands_str += f"    '{name}': ({flo:.2f}, {fhi:.2f}),\n"
    new_bands_str += "}"

    src_new = re.sub(
        r"SPECTRAL_BANDS\s*=\s*\{[^}]*\}",
        new_bands_str,
        src,
        flags=re.DOTALL,
    )
    with open(fe_path, 'w', encoding='utf-8') as f:
        f.write(src_new)
    print("\n✓ feature_engineering.py atualizado com novos SPECTRAL_BANDS")
else:
    print("\n(Edite _SAVE_BANDS = True para persistir)")

Bandas atuais vs calibradas:
  rot       atual=(0.5, 1.8)  novo=(0.1, 1.5)  ← ALTERADO
  vel1      atual=('?', '?')  novo=(3.0, 6.5)  ← ALTERADO
  vel2      atual=('?', '?')  novo=(6.5, 9.5)  ← ALTERADO
  vel3      atual=('?', '?')  novo=(9.5, 14.0)  ← ALTERADO
  noise     atual=(25.0, 50.0)  novo=(25.0, 50.0)  

(Edite _SAVE_BANDS = True para persistir)


## 4. Extração de Features (Temporal + Espectral)

In [5]:
rows_all = []

for cls in CLASSES_ORDER:
    df_cls = df_raw[df_raw['fan_state'] == cls].reset_index(drop=True)
    n = len(df_cls)
    if n < WINDOW_SIZE:
        print(f"  {cls}: insuficiente ({n} amostras)")
        continue

    # 1. Features temporais (amplitude absoluta)
    temp_rows = extract_features_windowed(
        df_cls, cls, AXES, WINDOW_SIZE, STEP_SIZE,
        timestamp_col='timestamp_s',
    )
    # 2. Features espectrais (peak_freq + band energies absolutas)
    spec_rows = extract_features_windowed_spectral(
        df_cls, cls, AXES, WINDOW_SIZE, STEP_SIZE,
        timestamp_col='timestamp_s', sampling_hz=SAMPLE_HZ, n_fft=N_FFT,
    )
    # 3. Features drift-resistant (fracoes, razoes, centroide, flatness)
    drift_rows = extract_features_windowed_drift_resistant(
        df_cls, cls, AXES, WINDOW_SIZE, STEP_SIZE,
        timestamp_col='timestamp_s', sampling_hz=SAMPLE_HZ, n_fft=N_FFT,
    )

    # 4. P1–P14 spectral moments (frequência × amplitude do espectro)
    mom_rows = extract_features_windowed_spectral_moments(
        df_cls, cls, AXES, WINDOW_SIZE, STEP_SIZE,
        timestamp_col='timestamp_s', sampling_hz=SAMPLE_HZ, n_fft=N_FFT,
    )

    meta_keys = {'fan_state','collection_id','window_start','window_end',
                 'timestamp_start','timestamp_end','timestamp_mean'}
    for tr, sr, dr, mr in zip(temp_rows, spec_rows, drift_rows, mom_rows):
        merged = {**tr,
                  **{k: v for k, v in sr.items() if k not in meta_keys},
                  **{k: v for k, v in dr.items() if k not in meta_keys},
                  **{k: v for k, v in mr.items() if k not in meta_keys}}
        rows_all.append(merged)

    print(f"  {cls:<20}  {len(temp_rows):>4} janelas  ({n:,} amostras)")

df_feat = pd.DataFrame(rows_all)
y = df_feat['fan_state'].values

feature_cols = [c for c in df_feat.columns
                if c not in {'fan_state','collection_id','window_start','window_end',
                             'timestamp_start','timestamp_end','timestamp_mean'}]
X = df_feat[feature_cols].fillna(0)

# Classificar features por tipo
_dr_keys  = ('_frac', '_hi_lo_ratio', '_centroid', '_spread', '_flatness', '_top2_freq_ratio')
_sp_keys  = ('_band_', '_peak_freq', '_peak_mag')
_mom_keys = ('_sp_p',)
drift_cols    = [c for c in feature_cols if any(k in c for k in _dr_keys)]
mom_cols      = [c for c in feature_cols if any(k in c for k in _mom_keys)]
spectral_cols = [c for c in feature_cols if any(k in c for k in _sp_keys)
                 and c not in drift_cols and c not in mom_cols]
temp_cols     = [c for c in feature_cols
                 if c not in drift_cols and c not in spectral_cols and c not in mom_cols]

print(f"\nDataset: {len(X)} janelas x {len(feature_cols)} features")
print(f"  Temporais (amplitude)      : {len(temp_cols)}")
print(f"  Espectrais (absolutas)     : {len(spectral_cols)}")
print(f"  Drift-resistant (razoes)   : {len(drift_cols)}")
print(f"  Momentos P1-P14 (espectro) : {len(mom_cols)}")
print(f"  Classes: {dict(pd.Series(y).value_counts())}")


  FAN_OFF: insuficiente (0 amostras)
  LOW_ROT_OFF: insuficiente (0 amostras)
  LOW_ROT_ON: insuficiente (0 amostras)
  MEDIUM_ROT_OFF: insuficiente (0 amostras)
  MEDIUM_ROT_ON: insuficiente (0 amostras)
  HIGH_ROT_OFF: insuficiente (0 amostras)
  HIGH_ROT_ON: insuficiente (0 amostras)


KeyError: 'fan_state'

## 5. Ranking ANOVA — Top Features por Separabilidade

In [ ]:
anova_rows = []
for col in feature_cols:
    groups = [X[col][y == c].values for c in CLASSES_ORDER if (y == c).sum() > 1]
    if len(groups) >= 2:
        f_stat, p_val = f_oneway(*groups)
        anova_rows.append({
            'feature': col,
            'f_stat': f_stat,
            'p_value': p_val,
            'type': 'spectral' if col in spectral_cols else 'temporal',
        })

df_anova = pd.DataFrame(anova_rows).sort_values('f_stat', ascending=False)
df_sig   = df_anova[df_anova['p_value'] < 0.05]

print(f"Features significativas (p<0.05): {len(df_sig)}/{len(df_anova)}")
print(f"  Temporais : {(df_sig['type']=='temporal').sum()}")
print(f"  Espectrais: {(df_sig['type']=='spectral').sum()}")
print(f"\nTop 25 features:")
print(df_sig.head(25)[['feature','f_stat','p_value','type']].to_string(index=False))

## 6. Treino GNB — 7 Classes (CV 5-fold)

In [ ]:
def remove_correlated(feat_list, X_df, threshold=0.85):
    if len(feat_list) < 2:
        return feat_list
    corr = X_df[feat_list].corr().abs()
    rank = {f: i for i, f in enumerate(df_anova['feature'].values)}
    to_drop = set()
    for i in range(len(feat_list)):
        for j in range(i + 1, len(feat_list)):
            if corr.iloc[i, j] > threshold:
                fi, fj = feat_list[i], feat_list[j]
                drop = fj if rank.get(fi, 9999) < rank.get(fj, 9999) else fi
                to_drop.add(drop)
    return [f for f in feat_list if f not in to_drop]

sig_set = set(df_sig['feature'])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Momentos P1-P14 por tipo de resistência ao drift
# Drift-resistant: P3, P4, P5, P7, P8, P9, P14
_DRIFT_SAFE_P = ('_sp_p3','_sp_p4','_sp_p5','_sp_p7','_sp_p8','_sp_p9','_sp_p14')
_ALL_P        = tuple(f'_sp_p{i}' for i in range(1, 15))

mom_dr_cols = [c for c in mom_cols if any(k in c for k in _DRIFT_SAFE_P)]
mom_all_cols = [c for c in mom_cols if any(k in c for k in _ALL_P)]

# A: temporais (amplitude) — baseline, sensivel ao drift
feat_A = [f for f in df_anova['feature'] if f in sig_set and f in temp_cols]
set_A  = remove_correlated(feat_A[:30], X)

# B: espectrais absolutas — peak_freq + band energies
feat_B = [f for f in df_anova['feature'] if f in sig_set and f in spectral_cols]
set_B  = remove_correlated(feat_B[:30], X)

# C: drift-resistant (fracoes, razoes, centroide) — sem momentos P
feat_C = [f for f in df_anova['feature'] if f in sig_set and f in drift_cols]
set_C  = remove_correlated(feat_C[:30], X)

# D: espectral + drift-resistant (sem temporais de amplitude)
feat_D = [f for f in df_anova['feature'] if f in sig_set and f not in temp_cols][:40]
set_D  = remove_correlated(feat_D, X)

# E: apenas peak_freq (puramente frequencial, maximo drift-safe)
set_E  = [f for f in feature_cols if '_peak_freq' in f]

# F: momentos P3/P4/P7/P8/P9/P14 — drift-resistant do espectro de amplitude
feat_F = [f for f in df_anova['feature'] if f in sig_set and f in mom_dr_cols]
set_F  = remove_correlated(feat_F[:30], X)

# G: todos os 14 momentos (inclui P1/P2/P6 — drift-sensitive para referência)
feat_G = [f for f in df_anova['feature'] if f in sig_set and f in mom_all_cols]
set_G  = remove_correlated(feat_G[:40], X)

# H: melhor combinacao — drift-resistant legacy + momentos P drift-safe
feat_H = [f for f in df_anova['feature']
          if f in sig_set and (f in drift_cols or f in mom_dr_cols)][:50]
set_H  = remove_correlated(feat_H, X)


# I: C + P9 isolado — testa se regularidade espectral resolve ROT_ON vs ROT_OFF
p9_cols = [f for f in mom_dr_cols if '_sp_p9' in f]
feat_I  = list(dict.fromkeys(set_C + [f for f in p9_cols if f in sig_set]))
set_I   = remove_correlated(feat_I, X)

# J: C + P4 — testa se kurtosis espectral detecta micro-diferença ROT_OFF
p4_cols = [f for f in mom_dr_cols if '_sp_p4' in f]
feat_J  = list(dict.fromkeys(set_C + [f for f in p4_cols if f in sig_set]))
set_J   = remove_correlated(feat_J, X)

results = {}
print(f"{'Conjunto':<35} {'CV Acc':>8}  {'+-':>6}  {'N':>4}  Obs")
print('─' * 72)
for name, fset, obs in [
    ('A  temporal (amplitude)',        set_A, 'DRIFT RISK'),
    ('B  espectral absoluta',          set_B, 'drift medio'),
    ('C  drift-resistant (legacy)',    set_C, 'DRIFT SAFE'),
    ('D  espectral + drift',           set_D, 'DRIFT SAFE'),
    ('E  peak_freq apenas',            set_E, 'DRIFT SAFE MAX'),
    ('F  momentos P3/4/7/8/9/14',      set_F, 'DRIFT SAFE'),
    ('G  todos 14 momentos',           set_G, 'drift misto'),
    ('H  drift-resistant + P moments', set_H, 'DRIFT SAFE'),
    ('I  C + P9 (regularity)',         set_I, 'DRIFT SAFE'),
    ('J  C + P4 (kurtosis)',            set_J, 'DRIFT SAFE'),
]:
    if not fset:
        print(f"  {name}: sem features significativas")
        continue
    scores = cross_val_score(GaussianNB(), X[fset], y, cv=cv, scoring='accuracy')
    results[name] = (scores, fset)
    print(f"  {name:<33} {scores.mean()*100:>7.2f}%  +-{scores.std()*100:.2f}%  "
          f"{len(fset):>3}  {obs}")

best_name     = max(results, key=lambda k: results[k][0].mean())
best_features = results[best_name][1]
print(f"\nMelhor conjunto: {best_name}")
print(f"Features ({len(best_features)}): {best_features}")


## 7. Matriz de Confusão — 7 Classes

In [ ]:
clf = GaussianNB()
clf.fit(X[best_features], y)
y_pred = clf.predict(X[best_features])

print(classification_report(y, y_pred, labels=CLASSES_ORDER, zero_division=0))

cm = confusion_matrix(y, y_pred, labels=CLASSES_ORDER)
fig = px.imshow(
    cm,
    x=CLASSES_ORDER, y=CLASSES_ORDER,
    text_auto=True, color_continuous_scale='Blues',
    labels=dict(x='Predito', y='Real'),
    title=f'Matriz de Confusão — {best_name} ({len(best_features)} features)',
)
fig.update_layout(height=500)
fig.write_html('output/figures/04_confusion_matrix_7class.html')
fig.show()

## 8. Export — Modelo GNB JSON (7 classes, 100 Hz)

In [ ]:
cv_scores = cross_val_score(clf, X[best_features], y, cv=cv)
train_acc = accuracy_score(y, y_pred)
ts = time.strftime('%Y%m%d_%H%M%S')

model_data = {
    'type':             'gaussian_nb',
    'version':          f'7class_100hz_{ts}',
    'generated_at':     time.strftime('%Y-%m-%d %H:%M:%S'),
    'sample_rate_hz':   SAMPLE_HZ,
    'window_size':      WINDOW_SIZE,
    'step_size':        STEP_SIZE,
    'n_fft':            N_FFT,
    'collection_id':    COLLECTION_ID,
    'features':         list(best_features),
    'labels':           list(clf.classes_),
    'priors':           {lbl: float(clf.class_prior_[i]) for i, lbl in enumerate(clf.classes_)},
    'stats':            {},
    'metrics': {
        'train_accuracy':   float(train_acc),
        'cv_accuracy_mean': float(cv_scores.mean()),
        'cv_accuracy_std':  float(cv_scores.std()),
    },
    'spectral_bands':   SPECTRAL_BANDS,
}

for i, lbl in enumerate(clf.classes_):
    model_data['stats'][lbl] = {
        feat: {'mean': float(clf.theta_[i, j]), 'var': float(clf.var_[i, j])}
        for j, feat in enumerate(best_features)
    }

date_str = time.strftime('%Y%m%d')
for path in [
    f'output/models/gnb_model_{date_str}.json',
    f'../models/gnb_model_{date_str}.json',
]:
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
    with open(path, 'w') as f:
        json.dump(model_data, f, indent=2)
    print(f"✓ Modelo exportado: {path}")

print(f"\nFeatures ({len(best_features)}): {best_features}")
print(f"Acurácia CV: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
print(f"\nPróximo passo:")
print(f"  1. Olhe os gráficos espectrais e calibre BANDS_CALIBRATED na Célula 3")
print(f"  2. Se SPECTRAL_BANDS mudar, re-execute a partir da Célula 4")
print(f"  3. Use o modelo exportado no backend e no JS do frontend")